# Série diária de qualidade do ar - Rio de Janeiro

## Objetivo
Este notebook consolida os dados diários das **17 estações** de qualidade do ar do município do Rio de Janeiro e gera uma série diária única para representar a cidade.

## Regra de agregação
- Para as variáveis numéricas, será aplicada a **média diária** entre estações.
- Para a variável **`chuva`**, será aplicada a **soma diária** entre estações.


## 1. Importações e configurações

Nesta seção definimos bibliotecas, lista de estações, template das URLs e parâmetros gerais do processamento.


In [13]:
from pathlib import Path
from typing import Dict, List

import pandas as pd

# Lista fixa com as 17 estações utilizadas no estudo.
ESTACOES: List[str] = [
    "bangu",
    "campo_grande",
    "centro",
    "copacabana",
    "iraja",
    "pedra_guaratiba",
    "sao_cristovao",
    "tijuca",
    "um_copacabana",
    "um_bangu",
    "um_caju",
    "um_centro",
    "um_del_castilho",
    "um_gamboa",
    "um_madureira",
    "um_maracana",
    "um_recreio",
]

# Template da URL pública dos arquivos tratados (um arquivo por estação).
URL_ESTACOES = (
    "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/"
    "refs/heads/Refactoring-And-Documentation/"
    "Data/IntermediaryData/MonitorAr/TreatedStations/{estacao}_tratado_diario.csv"
)

# Configurações principais do processamento.
COLUNA_DATA = "data"
COLUNA_CHUVA = "chuva"
COLUNAS_DESCARTAR = ["aqi", "qualidade_do_ar"]
CASAS_DECIMAIS = 3

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


## 2. Carregamento dos dados das estações

Leitura dos 17 arquivos diários, inclusão do identificador da estação e consolidação em um único DataFrame.


In [14]:
dados_estacoes = []
falhas_leitura = []

for estacao in ESTACOES:
    url = URL_ESTACOES.format(estacao=estacao)
    print(f"Lendo estação: {estacao} -> {url}")

    try:
        # Faz o parse da data já na leitura para reduzir retrabalho depois.
        df_estacao = pd.read_csv(url, parse_dates=[COLUNA_DATA])
        df_estacao["estacao"] = estacao
        dados_estacoes.append(df_estacao)
    except Exception as erro:
        falhas_leitura.append({"estacao": estacao, "url": url, "erro": str(erro)})

if not dados_estacoes:
    raise RuntimeError("Nenhum arquivo foi carregado. Verifique as URLs e a conexão.")

if falhas_leitura:
    print(f"Atenção: {len(falhas_leitura)} estação(ões) com erro de leitura.")
    display(pd.DataFrame(falhas_leitura))
else:
    print("Todas as estações foram carregadas com sucesso.")

# Consolida todos os registros das estações em uma única tabela.
df_estacoes = pd.concat(dados_estacoes, ignore_index=True)

# Remove colunas derivadas que não devem entrar na agregação final.
df_estacoes = df_estacoes.drop(columns=COLUNAS_DESCARTAR, errors="ignore")

print(f"Linhas totais consolidadas: {len(df_estacoes):,}")
print(f"Colunas disponíveis: {df_estacoes.columns.tolist()}")
display(df_estacoes.head())


Lendo estação: bangu -> https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/TreatedStations/bangu_tratado_diario.csv
Lendo estação: campo_grande -> https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/TreatedStations/campo_grande_tratado_diario.csv
Lendo estação: centro -> https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/TreatedStations/centro_tratado_diario.csv
Lendo estação: copacabana -> https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/TreatedStations/copacabana_tratado_diario.csv
Lendo estação: iraja -> https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/TreatedStations/iraja_tratado_diario.csv

,data,no,no2,so2,pm2_5,nox,ur,temp,o3,co,pm10,chuva,estacao
0,2012-01-01,2.452500,11.238750,0.470435,NaN,13.689167,90.205833,25.777083,29.651667,0.473333,20.312500,30.0,bangu
1,2012-01-02,4.909583,14.019583,0.396667,NaN,18.923750,94.847500,22.667917,19.629167,0.246250,5.416667,44.6,bangu
2,2012-01-03,7.156250,13.562083,0.518958,NaN,20.719583,74.360000,25.159583,23.405000,0.367292,21.208333,0.0,bangu
3,2012-01-04,14.143333,29.517083,0.770417,NaN,43.661667,70.591667,26.500417,29.161053,0.390417,40.875000,0.4,bangu
4,2012-01-05,6.610000,20.611250,0.337083,NaN,27.200000,70.697917,27.385417,45.120833,0.450000,28.625000,0.0,bangu


## 3. Padronização e preparação dos dados

Nesta etapa garantimos a granularidade diária na coluna de data e montamos o dicionário de agregação por variável.


In [15]:
# Garante tipo datetime e remove horário para manter granularidade diária.
df_estacoes[COLUNA_DATA] = pd.to_datetime(df_estacoes[COLUNA_DATA], errors="coerce").dt.normalize()

# Remove registros sem data válida, pois não podem ser agregados corretamente.
df_estacoes = df_estacoes.dropna(subset=[COLUNA_DATA]).copy()

# Identifica colunas numéricas que representarão os indicadores da cidade.
colunas_numericas = [
    coluna
    for coluna in df_estacoes.select_dtypes(include="number").columns
    if coluna != "estacao"
]

if not colunas_numericas:
    raise RuntimeError("Nenhuma coluna numérica foi encontrada para agregação.")

# Regra padrão: média diária para todas as variáveis numéricas.
agregacoes: Dict[str, str] = {coluna: "mean" for coluna in colunas_numericas}

# Regra de negócio: chuva deve ser soma diária, não média.
if COLUNA_CHUVA in agregacoes:
    agregacoes[COLUNA_CHUVA] = "sum"
else:
    print(
        f"A coluna '{COLUNA_CHUVA}' não foi encontrada entre as numéricas. "
        "A soma de chuva não será aplicada."
    )

print("Agregações definidas por coluna:")
display(pd.Series(agregacoes, name="agregacao"))


Agregações definidas por coluna:


no       mean
no2      mean
so2      mean
pm2_5    mean
nox      mean
ur       mean
temp     mean
o3       mean
co       mean
pm10     mean
chuva     sum
Name: agregacao, dtype: str

## 4. Geração da série diária do Rio de Janeiro

Agregação por dia para consolidar as medições das estações em uma série única do município.


In [16]:
serie_diaria_rj = (
    df_estacoes.groupby(COLUNA_DATA, as_index=False)
    .agg(agregacoes)
    .sort_values(COLUNA_DATA)
)

# Reordena colunas para manter a data no início da tabela final.
ordem_colunas = [COLUNA_DATA] + [
    coluna for coluna in serie_diaria_rj.columns if coluna != COLUNA_DATA
]
serie_diaria_rj = serie_diaria_rj[ordem_colunas]

# Arredonda colunas numéricas contínuas para facilitar leitura.
colunas_para_arredondar = serie_diaria_rj.select_dtypes(include="number").columns.tolist()
serie_diaria_rj[colunas_para_arredondar] = serie_diaria_rj[colunas_para_arredondar].round(CASAS_DECIMAIS)

print(f"Série diária gerada com {len(serie_diaria_rj):,} dias.")
display(serie_diaria_rj.head())


Série diária gerada com 2,557 dias.


,data,no,no2,so2,pm2_5,nox,ur,temp,o3,co,pm10,chuva
0,2012-01-01,5.762,22.376,2.667,14.508,28.102,92.521,25.556,22.905,0.397,22.838,228.2
1,2012-01-02,25.347,29.317,2.290,8.158,54.670,95.074,22.264,13.924,0.311,14.965,371.0
2,2012-01-03,23.367,28.593,3.704,9.875,51.883,73.642,25.091,15.997,0.243,26.374,0.2
3,2012-01-04,28.645,36.209,3.177,14.823,64.833,72.631,26.083,23.245,0.267,35.924,0.4
4,2012-01-05,17.859,31.785,3.113,11.958,49.605,74.860,26.661,36.343,0.250,31.761,0.0


## 5. Checagens rápidas de qualidade

Validações simples para conferir período temporal e valores ausentes.


In [17]:
data_inicio = serie_diaria_rj[COLUNA_DATA].min()
data_fim = serie_diaria_rj[COLUNA_DATA].max()

print(f"Período coberto: {data_inicio:%Y-%m-%d} até {data_fim:%Y-%m-%d}")
print(f"Total de dias na série: {serie_diaria_rj[COLUNA_DATA].nunique():,}")

print()
print("Valores ausentes por coluna:")
display(serie_diaria_rj.isna().sum().sort_values(ascending=False).to_frame("nulos"))

print()
print("Resumo estatístico das variáveis numéricas:")
display(serie_diaria_rj.describe().T)


Período coberto: 2012-01-01 até 2018-12-31
Total de dias na série: 2,557

Valores ausentes por coluna:


,nulos
pm2_5,41
no,23
so2,23
no2,23
o3,23
nox,23
ur,23
temp,23
pm10,23
co,23



Resumo estatístico das variáveis numéricas:


,count,mean,min,25%,50%,75%,max,std
data,2557,2015-07-02 00:00:00,2012-01-01 00:00:00,2013-10-01 00:00:00,2015-07-02 00:00:00,2017-04-01 00:00:00,2018-12-31 00:00:00,NaN
no,2534.0,16.331511,1.606,9.2425,13.3745,19.95975,88.887,11.092661
no2,2534.0,34.553219,6.954,26.75425,33.1745,40.312,88.473,11.174235
so2,2534.0,4.439855,0.228,2.53525,3.9155,5.87825,20.389,2.590891
pm2_5,2516.0,17.462048,0.333,10.208,14.938,21.81725,91.007,10.344109
nox,2534.0,50.875713,11.779,37.14925,46.854,59.95825,163.708,20.349592
ur,2534.0,69.889385,40.722,63.01525,69.9865,76.56275,97.247,10.085908
temp,2534.0,26.032219,15.897,23.595,25.818,28.4355,35.313,3.301615
o3,2534.0,30.135731,4.687,21.87125,28.986,37.0115,80.184,11.100493
co,2534.0,0.34352,0.084,0.261,0.321,0.394,1.203,0.126236


## 6. Exportação da série consolidada

Salva a série diária final no diretório intermediário do projeto.


In [18]:
def encontrar_raiz_projeto(inicio: Path) -> Path:
    # Procura a raiz subindo diretórios até encontrar simultaneamente Code e Data.
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "Code").exists() and (candidato / "Data").exists():
            return candidato
    return inicio


project_root = encontrar_raiz_projeto(Path.cwd())
output_dir = project_root / "Data" / "IntermediaryData" / "MonitorAr" / "DailyQualiarRj"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "serie_diaria_qualidade_ar_rio_de_janeiro.csv"
serie_diaria_rj.to_csv(output_file, index=False)

print(f"Arquivo salvo em: {output_file}")


Arquivo salvo em: c:\Users\jhter\OneDrive - cefet-rj.br\Projetos-2026\qualiar\Data\IntermediaryData\MonitorAr\DailyQualiarRj\serie_diaria_qualidade_ar_rio_de_janeiro.csv


## 7. Resultado final

A variável `serie_diaria_rj` contém a série diária consolidada da qualidade do ar do Rio de Janeiro e o CSV foi salvo em disco para consumo nas próximas etapas do pipeline.
